# Photons in the Sentinel-2C bands

[![Open in Google Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/OpenSkyLab-space/notebooks/blob/main/rope_and_photons_brno_en.ipynb)

**OpenSkyLab · notebook for the lab** [Rope and photons](https://openskylab.space/edu_viz_rope_and_photons.html) · [česká verze](rope_and_photons_brno_cs.ipynb)

In the lab you watched a wave on a rope. Its speed $v$, wavelength
$\lambda$ (the distance between neighbouring crests) and frequency $f$ (the number of
oscillations per second) are linked by

$$
v = \lambda f.
$$

**Light in vacuum obeys the same relation**, with the speed of light $c$. Light
carries energy in photons. From the wavelength we get the frequency and the energy
of one photon:

$$
f = \frac{c}{\lambda}, \qquad E = hf.
$$

$h$ is the Planck constant; we express energy in electronvolts (eV),
a unit of energy. **A shorter wavelength means a higher frequency and a larger photon
energy.** At the same frequency, more photons arriving on the same area per second
deliver more energy; the energy of each photon does not change.
Here you compute this for the real bands of the Sentinel-2C satellite.

**Goal:** from the metadata of a real Sentinel-2 scene, compute the frequency, photon energy and photon flux in each band and work out which detector each band needs.

**Data:** the product metadata of Sentinel-2C L1C, tile 33UXQ (around Brno), 5 February 2025 (`MTD_MSIL1C.xml` from the Google Cloud public archive): central wavelengths, spectral response of the bands and top-of-atmosphere solar irradiance. About 50 kB is downloaded.

**Tasks:** six. Write your answers in the *Your answer* cells below the tasks. Lines marked `# ◀ parameter` are meant to be changed; run the cell again with Shift + Enter.

**Time:** about 40 min. You need a Google account (Colab); nothing is installed on your computer. Run the cells in order from top to bottom.

## 1. Setup and data

**Required cell, run it first.** Loads the libraries and physical constants. The code is collapsed; reveal it with **Show code**.

In [ ]:
# @title Setup (run first) {display-mode: "form"}
import re

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests

# h: Planck constant, c: speed of light,
# e: elementary charge, k: Boltzmann constant (all SI)
from scipy.constants import c, e, h, k

> **💻 What the code does:** Downloads the metadata file (XML) of one real Sentinel-2 L1C scene.

In [ ]:
# metadata of one real Sentinel-2C L1C scene (tile 33UXQ, 5 February 2025)
URL = (
    "https://storage.googleapis.com/gcp-public-data-sentinel-2/tiles/33/U/XQ/"
    "S2C_MSIL1C_20250205T095231_N0511_R079_T33UXQ_20250205T115121.SAFE/MTD_MSIL1C.xml"
)
xml = requests.get(URL, timeout=60).text
print(f"downloaded {len(xml) / 1000:.0f} kB")

> **💻 What the code does:** Extracts from the XML, for each band, the name, central wavelength, spectral response in 1 nm steps and the top-of-atmosphere solar irradiance E0. It shows a table.

In [ ]:
# from the XML we take for each band: name, central wavelength, spectral response and solar irradiance E0
rows, srf = [], {}
for m in re.finditer(
    r'<Spectral_Information bandId="(\d+)" physicalBand="(\w+)">(.*?)</Spectral_Information>',
    xml,
    re.S,
):
    bid, name, body = int(m.group(1)), m.group(2), m.group(3)
    lmin = float(re.search(r'<MIN unit="nm">([\d.]+)', body).group(1))
    vals = np.array(re.search(r"<VALUES>(.*?)</VALUES>", body).group(1).split(), float)
    srf[name] = (lmin + np.arange(len(vals)), vals)  # 1 nm step
    e0 = float(
        re.search(rf'<SOLAR_IRRADIANCE bandId="{bid}"[^>]*>([\d.]+)', xml).group(1)
    )
    rows.append(
        {
            "band": name,
            "lambda_nm": float(
                re.search(r'<CENTRAL unit="nm">([\d.]+)', body).group(1)
            ),
            "E0_W_m2_um": e0,
        }
    )
bands = pd.DataFrame(rows).set_index("band")
bands

> **💻 What the code does:** Plots the spectral response of all bands against wavelength. It computes nothing.

In [ ]:
# spectral response of the bands: how strongly the detector responds to each wavelength
fig, ax = plt.subplots(figsize=(11, 3.2))
for name, (wl, r) in srf.items():
    ax.plot(wl, r, lw=1)
    ax.text(bands.lambda_nm[name], 1.03, name, ha="center", fontsize=8)
ax.set(
    xlabel="wavelength (nm)",
    ylabel="relative response",
    ylim=(0, 1.12),
    title="Sentinel-2C MSI: spectral response of the bands",
)
plt.show()

## 2. Wavelength and frequency

> **💻 What the code does:** From the central wavelength of each band it computes the frequency f = c/λ and, as a check, the product λ · f. It shows a table.

In [ ]:
lam = bands.lambda_nm * 1e-9  # m
bands["f_THz"] = c / lam / 1e12  # f = c / λ
bands["lambda_x_f"] = lam * bands.f_THz * 1e12
bands[["lambda_nm", "f_THz", "lambda_x_f"]].round(1)

> **✏️ Task 1.** The `lambda_x_f` column is the same for every band. What does it express and what does it correspond to on the rope in the lab? How many times higher is the frequency of band B2 (blue) than of B12 (SWIR 2)?

*Your answer:* 

## 3. Energy of one photon

The energy of one photon is given by the Planck–Einstein relation:

$$
E = hf = \frac{hc}{\lambda}.
$$

We use $h \approx 6.626\times10^{-34}\ \mathrm{J\,s}$,
$c \approx 3.00\times10^8\ \mathrm{m\,s^{-1}}$ and the wavelength
$\lambda$ in metres, so the result comes out in joules.
Since $1\ \mathrm{eV} = 1.602\,176\,634\times10^{-19}\ \mathrm{J}$,
we convert the energy to eV by dividing by this value. In the code it is the
numerical value of `e`, the elementary charge in coulombs, from `scipy.constants`.

> **💻 What the code does:** Computes the photon energy at the centre of each band in eV, plots it as a bar chart and prints the ratio of the blue and red photon energies.

In [ ]:
bands["E_photon_eV"] = h * c / lam / e  # E = h·f = h·c/λ, converted to electronvolts
ax = bands.E_photon_eV.plot.bar(figsize=(9, 3), color="tab:purple")
ax.set(ylabel="photon energy (eV)", title="Energy of one photon at the band centre")
plt.show()
print(
    f"blue (B2) / red (B4) photon: {bands.E_photon_eV['B2'] / bands.E_photon_eV['B4']:.2f} ×"
)

> **✏️ Task 2.** Check the energy ratio of the blue and red photon from the wavelengths alone, without constants. Why is the ratio of λ enough? How does the energy of one photon change when the blue light is twice as bright?

*Your answer:* 

## 4. From radiant energy to photon flux

The metadata give the top-of-atmosphere solar spectral irradiance $E_0$
in $\mathrm{W\,m^{-2}\,\mu m^{-1}}$. Dividing it by the energy of one photon
in joules gives the number of photons arriving per second on one square metre
per unit wavelength, here per micrometre:

$$
\Phi_\lambda = \frac{E_0}{E}.
$$

$\Phi_\lambda$ has the unit $\mathrm{s^{-1}\,m^{-2}\,\mu m^{-1}}$.
It is the spectral photon flux density, not the total number of photons in the band.

> **💻 What the code does:** Converts the photon energy from eV back to joules by multiplying by `e` and divides the solar spectral irradiance `E0` by it. It plots `E0` together with the spectral photon flux density (per second, per m² and per µm of spectrum) and shows a table.

In [ ]:
bands["photons_per_s_m2_um"] = bands.E0_W_m2_um / (bands.E_photon_eV * e)
fig, ax1 = plt.subplots(figsize=(9, 3.2))
ax1.plot(
    bands.lambda_nm, bands.E0_W_m2_um, "o-", c="tab:orange", label="E0 (W·m⁻²·µm⁻¹)"
)
ax2 = ax1.twinx()
ax2.plot(
    bands.lambda_nm,
    bands.photons_per_s_m2_um / 1e21,
    "s--",
    c="tab:blue",
    label="photons (10²¹ s⁻¹·m⁻²·µm⁻¹)",
)
ax1.set(xlabel="wavelength (nm)", ylabel="power")
ax2.set(ylabel="number of photons", ylim=(0, None))
ax1.set_ylim(0)
fig.legend(loc="lower left", bbox_to_anchor=(0.08, 0.15))
plt.show()
bands[["E0_W_m2_um", "E_photon_eV", "photons_per_s_m2_um"]].style.format("{:.3g}")

> **✏️ Task 3.** The power E0 falls more than twentyfold from blue (B2) to SWIR 2 (B12). Does the number of photons fall in the same way? Explain the difference. In which band does the most photons per micrometre fall on a square metre, and why is it not the band with the highest E0?

*Your answer:* 

## 5. Which bands a silicon chip can see

A photon frees an electron in a detector only if its energy is at least the band gap of the material. For silicon it is about 1.12 eV. We compute the cut-off wavelength and check how much of each band's response lies below it.

> **💻 What the code does:** From the band gap `Eg` it computes the cut-off wavelength of the detector and, for each band, the share of its spectral response below this limit. It shows a table.

In [ ]:
# Silicon 1.12; InGaAs about 0.75; HgCdTe 0.1–0.5 depending on composition.
Eg = 1.12  # ◀ parameter: band gap in eV
lam_cut = h * c / (Eg * e) * 1e9
print(f"cut-off wavelength: {lam_cut:.0f} nm")
share = {name: r[wl <= lam_cut].sum() / r.sum() for name, (wl, r) in srf.items()}
bands["share_detectable"] = pd.Series(share)
bands[["lambda_nm", "E_photon_eV", "share_detectable"]].style.format(
    {"lambda_nm": "{:.0f}", "E_photon_eV": "{:.2f}", "share_detectable": "{:.0%}"}
)

> **✏️ Task 4.** Which Sentinel-2 bands would a silicon detector capture fully, partly and not at all? Set `Eg = 0.75` (InGaAs) and run the cell again. What changed? Sentinel-2 uses HgCdTe detectors for bands B10 to B12: why?

*Your answer:* 

## 6. Thermal radiation and radar

We compare photons from a thermal band (10.9 µm, as in Landsat 8 TIRS) and from the Sentinel-1 radar (5.405 GHz) with the typical thermal energy in a detector at room temperature, k · T.

> **💻 What the code does:** Computes the photon energy for ionisation, the blue and SWIR bands, the thermal band and C-band radar and compares it with the thermal energy k · T at detector temperature `T`. It shows a table.

In [ ]:
T = 300  # ◀ parameter: detector temperature in kelvin
kT = k * T / e
other = pd.DataFrame(
    {
        "lambda_m": [h * c / (10 * e), 490e-9, 1610e-9, 10.9e-6, c / 5.405e9],
    },
    index=[
        "ionisation (10 eV)",
        "B2 blue",
        "B11 SWIR",
        "thermal 10.9 µm",
        "radar C 5.405 GHz",
    ],
)
other["E_eV"] = h * c / other.lambda_m / e
other["E_over_kT"] = other.E_eV / kT
print(f"k·T at {T} K = {kT * 1000:.1f} meV")
other.style.format({"lambda_m": "{:.3g}", "E_eV": "{:.3g}", "E_over_kT": "{:.3g}"})

> **✏️ Task 5.** A thermal photon has only about four times the energy of k · T, a radar photon is more than a thousand times weaker than k · T. Why are thermal cameras with photon detectors cooled, and why do a microbolometer and a radar antenna measure heating or the electric field of the wave rather than individual photons? Set `T = 77` (liquid nitrogen) and compare.

*Your answer:* 

> **✏️ Task 6.** At what wavelength does a photon have an energy of 10 eV (the approximate ionisation threshold)? Which part of the spectrum is it? Chemical bonds in molecules have energies of a few eV. Explain why a single UV photon can damage DNA while a strong radar signal carries more energy in total and still breaks no bond and ionises nothing.

*Your answer:* 

> **✏️ Extension (optional).** Classmates may have the notebook for another Sentinel-2 satellite (2A, 2B or 2C). Compare the central wavelengths of the bands and the E0 values: by how many nanometres do bands of the same name differ, and why must anyone who builds a time series from several satellites take this into account?

*Your answer:* 

---
### Data sources and references

- Copernicus Sentinel-2 data 2025, ESA; L1C metadata from the Google Cloud public archive (`gcp-public-data-sentinel-2`). Licence: [Legal notice on the use of Copernicus Sentinel Data](https://sentinels.copernicus.eu/documents/247904/690755/Sentinel_Data_Legal_Notice), free use with attribution.
- Constants h, c, e, k: exactly defined SI values since 2019, taken from `scipy.constants`; CODATA 2022 overview: [NIST](https://physics.nist.gov/cuu/pdf/wall_2022.pdf).
- Silicon band gap 1.12 eV at 300 K: Sze, S. M., Ng, K. K. (2007). *Physics of Semiconductor Devices*, 3rd ed. Wiley.
- Sentinel-1 radar frequency (C band) 5.405 GHz: ESA, Sentinel-1 User Handbook.

### How to cite

OpenSkyLab (2026). Photons in the Sentinel-2C bands [notebook, teaching resource], version 1.0. https://github.com/OpenSkyLab-space/notebooks/blob/main/rope_and_photons_brno_en.ipynb. Licence CC BY 4.0.

Notebook text and code: OpenSkyLab, licence [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/). Data are under their providers' licences (see above).

*This resource was developed by the OpenSkyLab team with the assistance of the generative AI Claude (Anthropic), used for drafting code and text. The authors are responsible for the content.*